# TP — Comptage de mots et comparaison des méthodes

**Nom :** Alex Abdallah  
**Cours :** Fondements et Applications de Big Data

## Objectif

Comparer trois méthodes de comptage des occurrences de mots :
1. Python séquentiel simple.
2. Pseudo-MapReduce en Python.
3. Hadoop MapReduce.

Les trois méthodes seront testées sur les mêmes fichiers,
avec des tailles croissantes.

In [1]:
from pathlib import Path
import random

dossier = Path("data")
dossier.mkdir(exist_ok=True)

N = 1_000
facteurs = [1, 10, 100, 1_000]

vocabulaire = [
    "chat", "chien", "arbre", "fleur", "maison",
    "soleil", "montagne", "livre", "ciel", "mer"
]

# Même fichier initial à chaque exécution.
generateur = random.Random(42)
lignes = [generateur.choice(vocabulaire) + "\n" for _ in range(N)]
bloc_initial = "".join(lignes)

fichiers = {}

for m in facteurs:
    chemin = dossier / f"mots_m{m}.txt"

    with chemin.open("w", encoding="utf-8") as fichier:
        for _ in range(m):
            fichier.write(bloc_initial)

    fichiers[m] = chemin

    print(
        f"{chemin.name} : {N * m:,} lignes, "
        f"{chemin.stat().st_size:,} octets"
    )

mots_m1.txt : 1,000 lignes, 7,117 octets
mots_m10.txt : 10,000 lignes, 71,170 octets
mots_m100.txt : 100,000 lignes, 711,700 octets
mots_m1000.txt : 1,000,000 lignes, 7,117,000 octets


## Méthode 1 — Python simple

Le programme lit le fichier et utilise un dictionnaire.
À chaque mot rencontré, son compteur augmente de 1.

In [2]:
def compter_python(chemin):
    with open(chemin, "r", encoding="utf-8") as fichier:
        mots = fichier.read().split()

    compteur = {}

    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1

    return compteur


resultat = compter_python(fichiers[1])
print(resultat)
print("Total :", sum(resultat.values()))

{'chien': 108, 'chat': 91, 'maison': 113, 'fleur': 116, 'arbre': 85, 'ciel': 103, 'mer': 95, 'montagne': 102, 'livre': 105, 'soleil': 82}
Total : 1000


## Méthode 2 — Pseudo-MapReduce Python

- Map : transforme chaque mot en une paire (mot, 1).
- Shuffle : regroupe les valeurs associées au même mot.
- Reduce : additionne ces valeurs.

Cette version s'exécute localement, sans parallélisation.

In [3]:
from collections import defaultdict


def mapper(ligne):
    return [(mot, 1) for mot in ligne.strip().split()]


def shuffle(paires):
    groupes = defaultdict(list)

    for mot, valeur in paires:
        groupes[mot].append(valeur)

    return groupes


def reducer(groupes):
    return {
        mot: sum(valeurs)
        for mot, valeurs in groupes.items()
    }


def compter_mapreduce(chemin):
    paires = []

    with open(chemin, "r", encoding="utf-8") as fichier:
        for ligne in fichier:
            paires.extend(mapper(ligne))

    return reducer(shuffle(paires))


resultat_mr = compter_mapreduce(fichiers[1])

assert resultat_mr == resultat
print("Les deux méthodes donnent les mêmes résultats.")

Les deux méthodes donnent les mêmes résultats.


In [4]:
from time import perf_counter
from statistics import median

mesures = []
reference = compter_python(fichiers[1])

methodes = {
    "Python simple": compter_python,
    "Pseudo-MapReduce": compter_mapreduce,
}

for m, chemin in fichiers.items():
    attendu = {
        mot: nombre * m
        for mot, nombre in reference.items()
    }

    for nom, fonction in methodes.items():
        durees = []

        for _ in range(3):
            debut = perf_counter()
            comptes = fonction(chemin)
            durees.append(perf_counter() - debut)

            assert comptes == attendu

        temps = median(durees)

        mesures.append({
            "m": m,
            "lignes": N * m,
            "octets": chemin.stat().st_size,
            "methode": nom,
            "temps_s": temps,
        })

        print(f"{N * m:>10,} lignes | {nom:<18} | {temps:.6f} s")

     1,000 lignes | Python simple      | 0.000633 s
     1,000 lignes | Pseudo-MapReduce   | 0.001514 s
    10,000 lignes | Python simple      | 0.003527 s
    10,000 lignes | Pseudo-MapReduce   | 0.008591 s
   100,000 lignes | Python simple      | 0.025055 s
   100,000 lignes | Pseudo-MapReduce   | 0.081694 s
 1,000,000 lignes | Python simple      | 0.271078 s
 1,000,000 lignes | Pseudo-MapReduce   | 0.925121 s


## Méthode 3 — Hadoop MapReduce avec Docker Compose

Hadoop est exécuté dans quatre conteneurs :

- NameNode : gère les métadonnées de HDFS.
- DataNode : stocke les blocs de données.
- ResourceManager : gère les ressources avec YARN.
- NodeManager : lance les tâches sur le nœud de calcul.

Les fichiers sont copiés dans HDFS avant les tests.
Le programme WordCount fourni avec Hadoop est utilisé
pour compter les occurrences des mots.

Tous les conteneurs fonctionnent sur un seul ordinateur.
Il s'agit donc d'un environnement Hadoop local à plusieurs
services, et non d'un cluster de plusieurs machines physiques.

### Protocole de mesure

Chaque fichier est testé trois fois.
La médiane des trois durées est retenue.

La durée mesurée inclut la commande Docker, la soumission
du job Hadoop et son exécution.

Elle exclut le téléchargement de l'image, le démarrage
des services, le transfert initial dans HDFS et la lecture
finale des résultats.

Les comptes sont vérifiés pour chaque mot. Pour un fichier
répété m fois, chaque compteur doit être multiplié par m.

In [5]:
%pip install pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path

print("Dossier courant :", Path.cwd())

assert Path("compose.yaml").exists(), (
    "Le dossier courant ne contient pas compose.yaml."
)

for variable in ["N", "fichiers", "mesures", "compter_python"]:
    assert variable in globals(), (
        f"La variable {variable} manque : "
        "réexécuter les cellules Python précédentes."
    )

print("Le notebook est prêt pour les tests Hadoop.")

Dossier courant : c:\Users\Alex\Documents\TP_ComptageMots
Le notebook est prêt pour les tests Hadoop.


In [7]:
import subprocess
import time
import uuid
from statistics import median
import pandas as pd

dossier_tp = Path.cwd()

dossier_resultats = dossier_tp / "resultats"
dossier_resultats.mkdir(exist_ok=True)

run_id = uuid.uuid4().hex[:8]

dossier_logs = dossier_resultats / f"logs_{run_id}"
dossier_logs.mkdir()

def docker_exec(*arguments):
    return subprocess.check_output(
        [
            "docker", "compose", "exec", "-T",
            "namenode", *arguments
        ],
        cwd=dossier_tp,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

# Enregistrer la version réellement utilisée.
version = docker_exec("hadoop", "version")
print(version)

(dossier_resultats / "version_hadoop.txt").write_text(
    version, encoding="utf-8"
)

# Enregistrer l'image exacte pour faciliter la reproduction.
image_info = subprocess.check_output(
    [
        "docker", "image", "inspect", "apache/hadoop:3",
        "--format", "{{json .RepoDigests}}"
    ],
    cwd=dossier_tp,
    text=True,
    encoding="utf-8",
    errors="replace",
)

(dossier_resultats / "image_hadoop.txt").write_text(
    image_info, encoding="utf-8"
)

reference = compter_python(fichiers[1])

mesures_hadoop = []
mesures_hadoop_brutes = []

for m, chemin in fichiers.items():
    attendu = {
        mot: nombre * m
        for mot, nombre in reference.items()
    }

    durees = []

    for repetition in range(1, 4):
        sortie = f"/tp/output_{run_id}_m{m}_r{repetition}"

        commande_hadoop = (
            "hadoop jar "
            "/opt/hadoop/share/hadoop/mapreduce/"
            "hadoop-mapreduce-examples-*.jar "
            f"wordcount /tp/input/mots_m{m}.txt {sortie}"
        )

        commande = [
            "docker", "compose", "exec", "-T", "namenode",
            "bash", "-lc", commande_hadoop
        ]

        print(
            f"Début : {N * m:,} lignes — "
            f"répétition {repetition}/3",
            flush=True
        )

        journal = dossier_logs / f"m{m}_r{repetition}.log"

        with journal.open("w", encoding="utf-8") as log:
            debut = time.perf_counter()

            execution = subprocess.run(
                commande,
                cwd=dossier_tp,
                stdout=log,
                stderr=subprocess.STDOUT,
            )

            duree = time.perf_counter() - debut

        if execution.returncode != 0:
            raise RuntimeError(
                f"Le job Hadoop a échoué. Ouvrir : {journal}"
            )

        # Récupération hors chronométrage.
        texte = docker_exec(
            "hdfs", "dfs", "-cat", f"{sortie}/part-r-*"
        )

        comptes = {}

        for ligne in texte.splitlines():
            mot, nombre = ligne.split("\t")
            comptes[mot] = int(nombre)

        assert comptes == attendu, (
            f"Les comptes sont incorrects pour m={m}."
        )

        (dossier_resultats / f"comptage_hadoop_m{m}.txt").write_text(
            texte, encoding="utf-8"
        )

        durees.append(duree)

        mesures_hadoop_brutes.append({
            "m": m,
            "repetition": repetition,
            "temps_s": duree,
            "correct": True,
        })

        # Sauvegarder chaque mesure réussie.
        pd.DataFrame(mesures_hadoop_brutes).to_csv(
            dossier_resultats / "temps_hadoop.csv",
            index=False
        )

        print(
            f"Terminé : {duree:.3f} s — comptes corrects",
            flush=True
        )

    mesures_hadoop.append({
        "m": m,
        "lignes": N * m,
        "methode": "Hadoop",
        "temps_s": median(durees),
    })

print("Les 12 tests Hadoop ont réussi.")

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /opt/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar

Début : 1,000 lignes — répétition 1/3
Terminé : 30.584 s — comptes corrects
Début : 1,000 lignes — répétition 2/3
Terminé : 27.430 s — comptes corrects
Début : 1,000 lignes — répétition 3/3
Terminé : 28.275 s — comptes corrects
Début : 10,000 lignes — répétition 1/3
Terminé : 27.784 s — comptes corrects
Début : 10,000 lignes — répétition 2/3
Terminé : 27.934 s — comptes corrects
Début : 10,000 lignes — répétition 3/3
Terminé : 28.124 s — comptes corrects
Début : 100,000 lignes — répétition 1/3
Terminé : 25.727 s — comptes corrects
Début : 100,000 lignes — répétition 2/3
Terminé : 26.729 s — comptes corrects
Début : 100,000 lignes —